# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

import base64
from io import BytesIO
from PIL import Image
from IPython.display import Audio, display

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()
openai.api_key = openai_api_key

OpenAI API Key exists and begins sk-proj-


In [3]:
def speak(text):
    """Function to speak the given text if TTS engine is available."""
    if engine:
        try:
            engine.say(text)
            engine.runAndWait()
        except Exception as e:
            print(f"Error in TTS processing: {e}")

In [4]:
# prompts

system_message = "You are a helpful German tutor who answers questions and help learning german language "
system_message += "and you create and answers question on German language A1 level."
system_message += "All your questions and information are related to Austria as we explore Austrian culture through a tutor."
system_message += "Give short, courteous answers. "
system_message += "Always be accurate. If you don't know the answer, say so."

In [5]:


def talker(message):
    response = openai.audio.speech.create(
        model="tts-1",
        voice="onyx",
        input=message)

    audio_stream = BytesIO(response.content)
    output_filename = "output_audio.mp3"
    with open(output_filename, "wb") as f:
        f.write(audio_stream.read())

    # Play the generated audio
    display(Audio(output_filename, autoplay=True))

talker("Well, hi there")

In [6]:

def speech_to_text(audio_file):
    """Convert speech to text using OpenAI Whisper"""
    try:
        if audio_file is None:
            return None
        
        with open(audio_file, "rb") as audio:
            transcript = openai.audio.transcriptions.create(
                model="whisper-1",
                file=audio
            )
        
        return transcript.text
    except Exception as e:
        print(f"Speech-to-text error: {str(e)}")
        return None

In [7]:

def process_audio_input(audio_file, chat_history):
    """Process audio input automatically"""
    if audio_file is None:
        return chat_history, None
    
    # Convert speech to text
    transcribed_text = speech_to_text(audio_file)

    print(transcribed_text)
    
    if transcribed_text is None:
        return chat_history, None

    chat_history += [{"role": "user", 'metadata': None, "content": transcribed_text, 'options': None}]
    
    # Automatically send to chat
    updated_history = chat(chat_history)
    
    return updated_history, None

In [8]:
def chat(history):
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]
    
    # Comment out or delete the next line if you'd rather skip Audio for now..
    talker(reply)
    
    return history

In [9]:
# More involved Gradio code as we're not using the preset Chat interface!
# Passing in inbrowser=True in the last line will cause a Gradio window to pop up immediately.

with gr.Blocks(theme=gr.themes.Soft(), css=".gradio-container {background-color: #f0f2f6}") as ui:
    gr.Markdown(
        """
        # 🎙️ Voice Chat with OpenAI

        Speak into the microphone to chat with the AI. You can also type your message in the textbox.
        You'll need to provide your own OpenAI API key.
        """
    )
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
    with gr.Row():
        entry = gr.Textbox(
            placeholder="Type your message here...",
            label="Chat with our AI Assistant:")

    with gr.Row():
        # Audio input component
        audio_input = gr.Audio(sources=["microphone"], type="filepath", label="Speak Here")
        
    with gr.Row():
        clear = gr.Button("Clear")

    def do_entry(message, history):
        history += [{"role":"user", "content":message}]
        return "", history

    entry.submit(do_entry, inputs=[entry, chatbot], outputs=[entry, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot]
    )

    # When the user stops recording, process the audio
    audio_input.stop_recording(
        process_audio_input,
        inputs=[audio_input, chatbot],
        outputs=[chatbot, audio_input]
    )
    
    clear.click(lambda: None, inputs=None, outputs=chatbot, queue=False)
    
ui.launch(inbrowser=True)




* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "C:\Users\anilt\anaconda3\envs\llms\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\anilt\anaconda3\envs\llms\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\anilt\anaconda3\envs\llms\Lib\site-packages\fastapi\applications.py", line 1054, in __call__
    await super().__call__(scope, receive, send)
  File "C:\Users\anilt\anaconda3\envs\llms\Lib\site-packages\starlette\applications.py", line 112, in __call__
    await self.middleware_stack(scope, receive, send)
  File "C:\Users\anilt\anaconda3\envs\llms\Lib\site-packages\starlette\middleware\errors.py", line 187, in __call__
    raise exc
  File "C

Hello, how are you? What is your name?
